In [1]:
from google.colab import drive
import torch
import sys
import numpy as np
import math
import time
import os
import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup
drive.mount('/content/gdrive', force_remount=True)


base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2"
# base_dir = "/content/gdrive/MyDrive/Final Project"
sys.path.append(base_dir)
from Models.Baseline_Model.GPT2_Baseline import GPT2_Baseline
from Models.Baseline_Model.train_Baseline import train_loop, estimate_loss
from Models.Configs import TrainConfig, BaselineConfig
from Datasets.DataLoader import CombinedBinDataLoader


device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")


MessageError: Error: credential propagation was unsuccessful

In [ ]:
#Load the hyperparameters, scheduler, etc... required for training
model_config = BaselineConfig()
train_config = TrainConfig()
eff_batch_size = train_config.batch_per_iter * train_config.grad_acc_factor
tokens_per_step = eff_batch_size * model_config.block_size


model = GPT2_Baseline(model_config, device)
model = model.to(device)
model = torch.compile(model)

optimizer = train_config.make_optimizer(model)
scheduler = train_config.make_scheduler(optimizer)
scaler = train_config.make_scaler()

get_lr = train_config.get_lr
torch.set_float32_matmul_precision('high')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
#Load the combined_bin dataset from drive and place it in Colab's files
!cp "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/Datasets/fineweb_1B.bin" "/content/fineweb_1B.bin"
train_loader, val_loader = CombinedBinDataLoader.create_loaders(
    '/content/fineweb_1B.bin',
    train_config.batch_per_iter,
    model_config.block_size,
    train_config,
    seed=42
)

Initialized loader with 57,983 chunks of size 16385.
Initialized loader with 3,052 chunks of size 16385.


In [ ]:
#Actually train the model
torch.cuda.empty_cache()
history = train_loop(model,
                     optimizer,
                     scheduler,
                     scaler,
                     device,
                     train_loader,
                     val_loader,
                     train_config,
                     model_config)

Trainable parameters: 124,046,592


W0424 04:48:23.379000 3576 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


Step    0 | Val Loss: 10.9687 | PPL: 58028.02
step    0/3623 | tokens 262,144 | Train loss 10.9700 | Time Since Last Train Print 35.7827 seconds
step   10/3623 | tokens 2,883,584 | Train loss 10.3965 | Time Since Last Train Print 72.6994 seconds
step   20/3623 | tokens 5,505,024 | Train loss 9.6338 | Time Since Last Train Print 73.5723 seconds
step   30/3623 | tokens 8,126,464 | Train loss 9.1742 | Time Since Last Train Print 73.5715 seconds
step   40/3623 | tokens 10,747,904 | Train loss 8.6932 | Time Since Last Train Print 72.9268 seconds
step   50/3623 | tokens 13,369,344 | Train loss 8.2147 | Time Since Last Train Print 72.5501 seconds
step   60/3623 | tokens 15,990,784 | Train loss 7.8037 | Time Since Last Train Print 72.2675 seconds
step   70/3623 | tokens 18,612,224 | Train loss 7.7448 | Time Since Last Train Print 71.8973 seconds
step   80/3623 | tokens 21,233,664 | Train loss 7.6345 | Time Since Last Train Print 71.8341 seconds
step   90/3623 | tokens 23,855,104 | Train loss 7

In [ ]:
message = model.infer("""One, two, three, """, 100, .8, 50)
print(message)

One, two, three, and four,000 people.
The American is an American player, and the most common ones of the country is in place, and the rest of the world is the best. The city is a powerful game, and the kids is the best. It is a very special story, and a lot of the most important character of the world.
The young is a very interesting, and if you can learn that it is not a good thing. The better, and the more impressive, and the


In [ ]:
#Save the model's state after traininf
path = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/ckpt_baseline_step_3623.pt"
torch.save({
            "step": 3623 ,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "history": history
        }, path)

In [ ]:
loss_fwd, ppl = estimate_loss(model, val_loader, device, 160)
print(f"Val FWD: {loss_fwd:.4f} | PPL: {ppl:.2f}")

Val FWD: 5.4499 | PPL: 232.75
